In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, to_timestamp

# 1. Xử lý bảng Orders (Ép kiểu ngày tháng)
df_orders = spark.table("bronze.olist_orders_dataset")

silver_orders = df_orders.select(
    col("order_id"),
    col("customer_id"),
    col("order_status"),
    to_timestamp(col("order_purchase_timestamp")).alias("purchase_ts"),
    to_timestamp(col("order_delivered_customer_date")).alias("delivered_ts")
).filter(col("order_id").isNotNull())

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver.orders")

# 2. Xử lý bảng Order Items
df_items = spark.table("bronze.olist_order_items_dataset")

silver_items = df_items.select(
    col("order_id"),
    col("product_id"),
    col("price").cast("double"),
    col("freight_value").cast("double")
)

silver_items.write.format("delta").mode("overwrite").saveAsTable("silver.order_items")

print("Hoàn thành tầng Silver!")